In [ ]:
# importing libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

In [ ]:
# loading dataset

df=pd.read_csv("../Data/train.csv")

In [ ]:
# EDA

print(df.head())

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
print(df.columns)

In [ ]:
df["Order Date"] = pd.to_datetime(
    df["Order Date"],
    format="%d/%m/%Y"
)

In [ ]:
df.set_index("Order Date", inplace=True)

daily_sales = df["Sales"].resample("D").sum()

print(daily_sales.head())
print(daily_sales.shape)

In [ ]:

plt.figure(figsize=(15,5))

plt.plot(daily_sales)

plt.title("Daily Sales Over Time")
plt.xlabel("Date")
plt.ylabel("Sales")

plt.show()

In [ ]:
rolling_mean = daily_sales.rolling(window=30).mean()

plt.figure(figsize=(15,5))

plt.plot(daily_sales, label="Original")
plt.plot(rolling_mean, label="30-Day Moving Average")

plt.legend()
plt.show()

In [ ]:
monthly_sales = daily_sales.resample("ME").sum()

monthly_sales.head()

In [ ]:
yearly_sales = daily_sales.resample("YE").sum()

In [ ]:
daily_sales = pd.DataFrame(daily_sales)


# creating features

daily_sales["Lag_1"] = daily_sales["Sales"].shift(1)

In [ ]:
daily_sales["Lag_7"] = daily_sales["Sales"].shift(7)

In [ ]:
daily_sales["Lag_30"] = daily_sales["Sales"].shift(30)

In [ ]:
daily_sales.head(35)

In [ ]:
daily_sales["Day"] = daily_sales.index.day
daily_sales["Month"] = daily_sales.index.month
daily_sales["Year"] = daily_sales.index.year
daily_sales["DayOfWeek"] = daily_sales.index.dayofweek

In [ ]:
daily_sales.isnull().sum()

In [ ]:
daily_sales.dropna(inplace=True)

In [ ]:
daily_sales.info()

In [ ]:
X = daily_sales.drop("Sales", axis=1)
y = daily_sales["Sales"]

In [ ]:
train_size = int(len(daily_sales) * 0.8)

train = daily_sales[:train_size]

test = daily_sales[train_size:]

In [ ]:
X_train = train.drop("Sales", axis=1)

y_train = train["Sales"]

X_test = test.drop("Sales", axis=1)

y_test = test["Sales"]

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

In [ ]:
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

In [ ]:
mae = mean_absolute_error(
    y_test,
    y_pred
)

print("MAE:", mae)

In [ ]:
rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

print("RMSE:", rmse)

In [ ]:
print(daily_sales["Sales"].mean())

In [ ]:
importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.bar(
    importance["Feature"],
    importance["Importance"]
)

plt.xticks(rotation=45)

plt.show()

In [ ]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, 15],
    "min_samples_split": [2, 5]
}
grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid.fit(X_train, y_train)

In [ ]:
print(grid.best_params_)

In [ ]:
best_model = grid.best_estimator_

In [ ]:
y_pred = best_model.predict(X_test)

In [ ]:
mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

print("MAE:", mae)
print("RMSE:", rmse)

In [ ]:
results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})

results.head(10)

In [ ]:
results["Error"] = (
    results["Actual"]
    - results["Predicted"]
)

results["Absolute_Error"] = (
    results["Error"].abs()
)

results.sort_values(
    by="Absolute_Error",
    ascending=False
).head(10)

In [ ]:
sample = X_test.iloc[[0]]

prediction = best_model.predict(sample)

print("Actual:", y_test.iloc[0])

print("Predicted:", prediction[0])

In [ ]:
import joblib

joblib.dump(
    best_model,
    "../sales_forecasting_model.pkl"
)